In [ ]:
import pandas as pd

# Load datasets
df1 = pd.read_csv("Audible_Catlog.csv")
df2 = pd.read_csv("Audible_Catlog_Advanced_Features.csv")

# Merge on Book Name + Author
books = pd.merge(df1, df2, on=["Book Name", "Author"], how="outer")

# Handle missing values
books['Rating'] = books['Rating'].fillna(books['Rating'].mean())
books['Number of Reviews'] = books['Number of Reviews'].fillna(0)
books['Genre'] = books['Genre'].fillna("Unknown")

# Remove duplicates
books = books.drop_duplicates(subset=["Book Name", "Author"])

print(books.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Popular genres
genre_counts = books['Genre'].value_counts()
plt.figure(figsize=(10,5))
sns.barplot(x=genre_counts.index[:10], y=genre_counts.values[:10])
plt.title("Top 10 Genres")
plt.xticks(rotation=45)
plt.show()

# Ratings distribution
plt.figure(figsize=(8,5))
sns.histplot(books['Rating'], bins=20, kde=True)
plt.title("Rating Distribution")
plt.show()

# Correlation heatmap
plt.figure(figsize=(8,6))
sns.heatmap(books[['Rating','Number of Reviews','Price']].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Use book descriptions for NLP
tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf_matrix = tfidf.fit_transform(books['Description'].fillna(""))

print("TF-IDF shape:", tfidf_matrix.shape)

In [ ]:
from sklearn.cluster import KMeans

# Cluster books based on description features
kmeans = KMeans(n_clusters=10, random_state=42)
books['Cluster'] = kmeans.fit_predict(tfidf_matrix)

# Check cluster distribution
print(books['Cluster'].value_counts())

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Function to recommend books
def recommend_books(title, n=5):
    if title not in books['Book Name'].values:
        return "Book not found!"
    idx = books[books['Book Name'] == title].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_books = [books.iloc[i[0]]['Book Name'] for i in sim_scores[1:n+1]]
    return top_books

print(recommend_books("The Hobbit"))